# Two-material topology optimization with void

This notebook implements a two-dimensional cantilever example using the
modified SIMP interpolation for two elastic materials and void described in
Section 2.9.3 of Bendsøe and Sigmund (2004).

> Bendsøe, M. P., and Sigmund, O. *Topology Optimization: Theory, Methods,
> and Applications*. Springer, 2004, Section 2.9.3.

In [ ]:
import io
from contextlib import redirect_stdout
from pathlib import Path

import jax
import jax.numpy as np
import matplotlib.pyplot as plt
import numpy as onp
from IPython.display import Image as DisplayImage
from IPython.display import display
from jax_fem import logger
from jax_fem.generate_mesh import Mesh, get_meshio_cell_type, rectangle_mesh
from jax_fem.solver import ad_wrapper
from matplotlib.patches import Patch
from PIL import Image as PILImage

from topax.filter import build_conv_filter
from topax.optimizer import MMA
from topax.problem import TopOptProblem

jax.config.update('jax_enable_x64', True)
logger.setLevel('WARNING')

## 1. Cantilever model

The left edge is fixed and a downward unit load is applied at the midpoint
of the right edge. Material 1 is the stiff red phase, material 2 is the
softer blue phase, and the remaining domain is void.

In [ ]:
class MultiMaterialElasticity(TopOptProblem):
    """Plane-stress problem with two materials and void."""

    def custom_init(self):
        self.fe = self.fes[0]
        self.fe.flex_inds = np.arange(self.fe.num_cells)

    def get_tensor_map(self):
        def stress(u_grad, rho_1, rho_2, penal_1, penal_2):
            E_1 = 1.0
            E_2 = 0.2
            E_min = 1e-9
            E_mix = rho_2[0] ** penal_2[0] * E_1 + (
                1.0 - rho_2[0] ** penal_2[0]
            ) * E_2
            E = E_min + rho_1[0] ** penal_1[0] * (E_mix - E_min)

            nu = 0.3
            mu = E / (2.0 * (1.0 + nu))
            lam = E * nu / ((1.0 + nu) * (1.0 - 2.0 * nu))
            lam = 2.0 * mu * lam / (lam + 2.0 * mu)
            strain = 0.5 * (u_grad + u_grad.T)
            return lam * np.trace(strain) * np.eye(2) + 2.0 * mu * strain

        return stress

    def set_params(self, params):
        rho_1, rho_2, penal_1, penal_2 = params
        num_quads = self.fe.num_quads
        self.internal_vars = [
            np.repeat(rho_1[:, None, :], num_quads, axis=1),
            np.repeat(rho_2[:, None, :], num_quads, axis=1),
            penal_1 * np.ones((self.fe.num_cells, num_quads, 1)),
            penal_2 * np.ones((self.fe.num_cells, num_quads, 1)),
        ]

    def compliance(self, displacement):
        return np.sum(self.point_force * displacement[self.load_node])


def prepare_problem(Nx, Ny, Lx, Ly):
    """Build the cantilever finite-element problem."""
    ele_type = 'QUAD4'
    cell_type = get_meshio_cell_type(ele_type)
    meshio_mesh = rectangle_mesh(
        Nx=Nx,
        Ny=Ny,
        domain_x=Lx,
        domain_y=Ly,
    )
    mesh = Mesh(meshio_mesh.points, meshio_mesh.cells_dict[cell_type])

    def fixed_location(point):
        return np.isclose(point[0], 0.0, atol=1e-8)

    def load_location(point):
        return np.logical_and(
            np.isclose(point[0], Lx, atol=1e-8),
            np.isclose(point[1], Ly / 2.0, atol=1e-8),
        )

    zero = lambda point: 0.0
    problem = MultiMaterialElasticity(
        mesh,
        vec=2,
        dim=2,
        ele_type=ele_type,
        dirichlet_bc_info=[
            [fixed_location, fixed_location],
            [0, 1],
            [zero, zero],
        ],
    )
    problem.point_force = np.array([0.0, -1.0])
    problem.load_node = problem.add_point_load(
        load_location,
        problem.point_force,
    )

    fwd_pred = ad_wrapper(
        problem,
        solver_options={'linear': True, 'spsolve_solver': {}},
        adjoint_solver_options={'spsolve_solver': {}},
    )
    return fwd_pred, problem


Nx, Ny = 80, 40
Lx, Ly = 2.0, 1.0
fwd_pred, problem = prepare_problem(Nx, Ny, Lx, Ly)

## 2. Material interpolation and constraints

Each element has two filtered design variables. The first determines
material versus void, while the second selects between the two materials:

$$
\phi_{1,e}=\rho_e^{(1)}\rho_e^{(2)},\qquad
\phi_{2,e}=\rho_e^{(1)}(1-\rho_e^{(2)}),\qquad
\phi_{0,e}=1-\rho_e^{(1)}.
$$

The modified SIMP interpolation is

$$
E_e=(\rho_e^{(1)})^{p_1}
\left[(\rho_e^{(2)})^{p_2}E_1+
\left(1-(\rho_e^{(2)})^{p_2}\right)E_2\right].
$$

Both material volume fractions are limited to 25 percent.

In [ ]:
volume_1_limit = 0.25
volume_2_limit = 0.25
rmin = 2.4

H, Hs = build_conv_filter(problem, rmin=rmin)


def physical_design(design):
    return H @ design / Hs


def phase_fractions(design):
    rho = physical_design(design)
    material_1 = rho[:, 0] * rho[:, 1]
    material_2 = rho[:, 0] * (1.0 - rho[:, 1])
    void = 1.0 - rho[:, 0]
    return rho, material_1, material_2, void


def compliance(design, penal_1, penal_2):
    rho, material_1, material_2, void = phase_fractions(design)
    displacement = fwd_pred(
        (rho[:, 0:1], rho[:, 1:2], penal_1, penal_2)
    )[0]
    objective = problem.compliance(displacement)
    return objective, (material_1, material_2, void)


def volume_constraints(design):
    _, material_1, material_2, _ = phase_fractions(design)
    return np.array([
        np.mean(material_1) / volume_1_limit - 1.0,
        np.mean(material_2) / volume_2_limit - 1.0,
    ])


print('Material limits:')
print('  void:       E = 1e-9')
print('  blue phase: E = 0.2')
print('  red phase:  E = 1.0')

## 3. Gradient check

In [ ]:
initial_design = 0.5 * np.ones((Nx * Ny, 2))
initial_penal = np.array(1.0)

(initial_objective, _), initial_gradient = jax.value_and_grad(
    compliance,
    has_aux=True,
)(initial_design, initial_penal, initial_penal)

direction = np.sin(np.arange(initial_design.size)).reshape(initial_design.shape)
direction = direction / np.linalg.norm(direction)
epsilon = 1e-4
objective_plus = compliance(
    initial_design + epsilon * direction,
    initial_penal,
    initial_penal,
)[0]
objective_minus = compliance(
    initial_design - epsilon * direction,
    initial_penal,
    initial_penal,
)[0]
finite_difference = (objective_plus - objective_minus) / (2.0 * epsilon)
automatic_derivative = np.sum(initial_gradient * direction)
relative_error = np.abs(
    finite_difference - automatic_derivative
) / np.maximum(np.abs(finite_difference), 1e-12)

print(f'Initial compliance: {float(initial_objective):.6f}')
print(f'Finite difference:  {float(finite_difference):.8f}')
print(f'AD derivative:      {float(automatic_derivative):.8f}')
print(f'Relative error:     {float(relative_error):.3e}')

## 4. MMA optimization

In [ ]:
def run_optimization(max_iterations=250, frame_stride=2):
    design = initial_design.copy()
    optimizer = MMA(move=0.1, c_penalty=1e5, d_penalty=0.0)
    penal_1 = 1.0
    penal_2 = 1.0
    records = []
    frames = []

    for iteration in range(1, max_iterations + 1):
        with redirect_stdout(io.StringIO()):
            (
                objective,
                (material_1, material_2, void),
            ), objective_grad = jax.value_and_grad(
                compliance,
                has_aux=True,
            )(
                design,
                np.array(penal_1),
                np.array(penal_2),
            )

        constraints = volume_constraints(design)
        constraint_grads = jax.jacrev(volume_constraints)(design)
        volume_1 = float(np.mean(material_1))
        volume_2 = float(np.mean(material_2))

        old_design = design.copy()
        if iteration < max_iterations:
            design = np.asarray(
                optimizer.update(
                    old_design,
                    objective,
                    objective_grad,
                    constraints,
                    constraint_grads,
                )
            )
        change = float(np.max(np.abs(design - old_design)))

        record = {
            'iteration': iteration,
            'objective': float(objective),
            'volume_1': volume_1,
            'volume_2': volume_2,
            'change': change,
            'penal_1': penal_1,
            'penal_2': penal_2,
        }
        records.append(record)

        if (
            iteration == 1
            or iteration % frame_stride == 0
            or iteration == max_iterations
        ):
            frames.append({
                **record,
                'material_1': onp.asarray(material_1),
                'material_2': onp.asarray(material_2),
                'void': onp.asarray(void),
            })

        print(
            f"It.:{iteration:4d}, Obj.:{float(objective):10.4f}, "
            f"V1:{volume_1:7.4f}, V2:{volume_2:7.4f}, "
            f"ch.:{change:7.4f}, p1:{penal_1:3.1f}, p2:{penal_2:3.1f}"
        )

        penal_1 = min(penal_1 + 0.1, 3.0)
        penal_2 = min(penal_2 + 0.1, 3.0)

    return {
        'design': onp.asarray(design),
        'records': records,
        'frames': frames,
    }


result = run_optimization()

## 5. Save and visualize the material distribution

In [ ]:
red = onp.array([0.86, 0.10, 0.12])
blue = onp.array([0.10, 0.27, 0.90])
white = onp.ones(3)


def phase_image(frame):
    colors = (
        frame['void'][:, None] * white
        + frame['material_1'][:, None] * red
        + frame['material_2'][:, None] * blue
    )
    return colors.reshape(Nx, Ny, 3).transpose(1, 0, 2)


frames = result['frames']
records = result['records']
output_data = Path('docs/data/example_topopt_multimaterial.npz')
output_data.parent.mkdir(parents=True, exist_ok=True)
onp.savez_compressed(
    output_data,
    design=result['design'],
    frame_iteration=onp.array([frame['iteration'] for frame in frames]),
    frame_objective=onp.array([frame['objective'] for frame in frames]),
    frame_volume_1=onp.array([frame['volume_1'] for frame in frames]),
    frame_volume_2=onp.array([frame['volume_2'] for frame in frames]),
    frame_material_1=onp.stack([frame['material_1'] for frame in frames]),
    frame_material_2=onp.stack([frame['material_2'] for frame in frames]),
    frame_void=onp.stack([frame['void'] for frame in frames]),
    record_iteration=onp.array([record['iteration'] for record in records]),
    record_objective=onp.array([record['objective'] for record in records]),
    record_volume_1=onp.array([record['volume_1'] for record in records]),
    record_volume_2=onp.array([record['volume_2'] for record in records]),
    record_change=onp.array([record['change'] for record in records]),
    record_penal_1=onp.array([record['penal_1'] for record in records]),
    record_penal_2=onp.array([record['penal_2'] for record in records]),
)


def add_material_legend(ax, fontsize):
    handles = [
        Patch(facecolor=red, edgecolor='none', label='Material 1'),
        Patch(facecolor=blue, edgecolor='none', label='Material 2'),
        Patch(facecolor=white, edgecolor='0.6', label='Void'),
    ]
    ax.legend(
        handles=handles,
        loc='upper center',
        bbox_to_anchor=(0.5, -0.05),
        ncol=3,
        frameon=False,
        fontsize=fontsize,
    )


final_frame = frames[-1]


def render_frame(frame):
    with plt.rc_context({
        'text.usetex': True,
        'font.family': 'serif',
        'font.size': 22,
    }):
        fig, ax = plt.subplots(figsize=(12, 6), dpi=150)
        ax.imshow(
            phase_image(frame),
            origin='lower',
            extent=(0.0, Lx, 0.0, Ly),
            interpolation='nearest',
        )
        ax.set_aspect('equal')
        ax.set_axis_off()
        ax.set_title(
            rf"$i={frame['iteration']}\quad "
            rf"c={frame['objective']:.3f}\quad "
            rf"V_1/V_0={100.0 * frame['volume_1']:.1f}\%\quad "
            rf"V_2/V_0={100.0 * frame['volume_2']:.1f}\%\quad "
            rf"p_1=p_2={frame['penal_1']:.1f}$",
            fontsize=25,
            pad=12,
        )
        add_material_legend(ax, fontsize=22)
        fig.subplots_adjust(left=0.01, right=0.99, bottom=0.16, top=0.88)
        fig.canvas.draw()
        rgba = onp.asarray(fig.canvas.buffer_rgba()).copy()
        plt.close(fig)
    return PILImage.fromarray(rgba)


images = [render_frame(frame) for frame in frames]
gif_path = Path('docs/imgs/example_topopt_multimaterial.gif')
images[0].save(
    gif_path,
    save_all=True,
    append_images=images[1:],
    duration=120,
    loop=0,
    optimize=False,
)
display(DisplayImage(filename=str(gif_path)))

print(f"Final compliance: {final_frame['objective']:.6f}")
print(f"Material 1 volume fraction: {final_frame['volume_1']:.6f}")
print(f"Material 2 volume fraction: {final_frame['volume_2']:.6f}")